# Salinity

In [32]:
from pathlib import Path
import re

# Folder path
folder = Path(r"N:\Deltabox\Postbox\Athanasiou, Panos\van_Sepehr\salinity_mekong\projections_gridded")

# Initialize lists
cc85_files = []
cc45_files = []

# Collect .xyz files
for file in folder.glob("*.xyz"):
    fname = file.name.lower()
    if "_cc85" in fname:
        cc85_files.append(file)
    elif "_cc45" in fname:
        cc45_files.append(file)

# Sort alphabetically
cc85_files = sorted(cc85_files, key=lambda x: x.name)
cc45_files = sorted(cc45_files, key=lambda x: x.name)

# Function to clean filenames
def clean_name(filename, pattern):
    # Remove everything before and including the pattern, plus underscores
    name = re.sub(rf".*{pattern}_*", "", filename)
    # Remove trailing number before the extension (e.g. _2050 or 2050)
    name = re.sub(r"[_\-]?\d+(?=\.\w+$)", "", name)
    return name

# Clean names
cc85_cleaned = [clean_name(f.name, "_cc85") for f in cc85_files]
cc45_cleaned = [clean_name(f.name, "_cc45") for f in cc45_files]

# Remove duplicates (after cleaning)
cc85_unique = sorted(set(cc85_cleaned))
cc45_unique = sorted(set(cc45_cleaned))

# Compare sets
set85 = set(cc85_unique)
set45 = set(cc45_unique)

unique_to_85 = sorted(set85 - set45)
unique_to_45 = sorted(set45 - set85)
common_files = sorted(set85 & set45)

# Print results
print("✅ Unique to cc85:")
for name in unique_to_85:
    print(name)

print("\n✅ Unique to cc45:")
for name in unique_to_45:
    print(name)

print(f"\n✅ Common to both lists: {len(common_files)}")
for name in common_files:
    print(name)


✅ Unique to cc85:
blsb2rb3y.xyz
ndsb2rb3y.xyz
qd10sb2rb3y.xyz
qd20sb2rb3y.xyz
sb2y.xyz
slrsb2rb3y.xyz
slrsb2y.xyz
slry.xyz
wdsb2rb3y.xyz

✅ Unique to cc45:
blsm2rb1y.xyz
ndsm2rb1y.xyz
qd10sm2rb1y.xyz
qd20sm2rb1y.xyz
sm2rb1y.xyz
sm2y.xyz
wdsm2rb1y.xyz

✅ Common to both lists: 2
tay.xyz
y.xyz


In [ ]:
from pathlib import Path
import re
import shutil
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.shutil import copy as rio_copy

def process_xyz_to_geospatial(file_path):
    # Read the xyz file into a pandas DataFrame
    df = pd.read_csv(file_path, delim_whitespace=True, header=None, names=['x', 'y', 'z'])

    # Create geometry column (2D point)
    df['geometry'] = df.apply(lambda row: Point(row['x'], row['y']), axis=1)

    # Convert to GeoDataFrame and set CRS to WGS-84 / UTM 48N
    gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:32648")

    # --- Create raster before dropping NaNs ---
    df_raster = df.dropna(subset=['z'])

    # Define raster resolution
    resolution = 2000  # meters

    # Compute bounds
    xmin, ymin, xmax, ymax = df_raster['x'].min(), df_raster['y'].min(), df_raster['x'].max(), df_raster['y'].max()

    # Define raster size
    width = int(np.ceil((xmax - xmin) / resolution))
    height = int(np.ceil((ymax - ymin) / resolution))

    # Create affine transform
    transform = from_origin(xmin, ymax, resolution, resolution)

    # Create empty raster array
    raster = np.full((height, width), np.nan)

    # Map points to raster cells
    for _, row in df_raster.iterrows():
        col = int((row['x'] - xmin) // resolution)
        row_idx = int((ymax - row['y']) // resolution)
        if 0 <= row_idx < height and 0 <= col < width:
            raster[row_idx, col] = row['z']

    # Save initial GeoTIFF
    temp_tif = file_path.replace('.xyz', '_temp.tif')
    with rasterio.open(
        temp_tif,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=raster.dtype,
        crs="EPSG:32648",
        transform=transform,
        nodata=np.nan,
        compress='LZW'
    ) as dst:
        dst.write(raster, 1)

    # Convert to Cloud Optimized GeoTIFF
    cog_tif = file_path.replace('.xyz', '.tif')
    rio_copy(temp_tif, cog_tif, copy_src_overviews=True, driver='COG', compress='LZW')

    # Remove temporary file
    Path(temp_tif).unlink() 

    # Remove .xyz file
    Path(file_path).unlink()

    # Remove any .xml in the folder
    for xml_file in Path(file_path).parent.glob("*.xml"):
        xml_file.unlink()

    print(f"Cloud Optimized GeoTIFF saved to {cog_tif}")

    # --- Continue with GeoJSON ---
    gdf.dropna(inplace=True)
    output_path = file_path.replace('.xyz', '.geojson')
    # gdf.to_file(output_path, driver='GeoJSON')

    print(f"GeoJSON saved to {output_path}")

    return cog_tif

# Base folder
base_folder = Path(r"N:\Deltabox\Postbox\Athanasiou, Panos\van_Sepehr\salinity_mekong\projections_gridded")

# List to store copied file paths
copied_files = []

# Create output folders and copy files based on pattern
for file in base_folder.glob("*.xyz"):
    fname = file.name

    # Extract the pattern: "_cc85..." or "_cc45..." up to the last letter before the number
    match = re.search(r"_(cc\d{2}[a-z0-9]+?)(?=\d+\.xyz$)", fname.lower())
    if not match:
        print(f"⚠️ Skipping file (no match found): {fname}")
        continue

    folder_name = match.group(1)  # e.g. cc85sb2y
    target_folder = base_folder / folder_name
    target_folder.mkdir(exist_ok=True)

    # Copy the file to the new folder
    target_path = target_folder / file.name
    shutil.copy2(file, target_path)
    copied_files.append(str(target_path))  # Add the copied file path to the list
    print(f"✅ Copied: {file.name} → {folder_name}/")

print("\n🎉 Done organizing files!")
print(f"\n📄 List of copied files ({len(copied_files)}):")
for f in copied_files:
    print(f)

cog_files = []

for file_path in copied_files:
    cog_file = process_xyz_to_geospatial(file_path)
    cog_files.append(cog_file)

✅ Copied: P50_cc45blsm2rb1y40.xyz → cc45blsm2rb1y/
✅ Copied: P50_cc45ndsm2rb1y40.xyz → cc45ndsm2rb1y/
✅ Copied: P50_cc45qd10sm2rb1y40.xyz → cc45qd10sm2rb1y/
✅ Copied: P50_cc45qd20sm2rb1y40.xyz → cc45qd20sm2rb1y/
✅ Copied: P50_cc45sm2rb1y30.xyz → cc45sm2rb1y/
✅ Copied: P50_cc45sm2rb1y40.xyz → cc45sm2rb1y/
✅ Copied: P50_cc45sm2rb1y50.xyz → cc45sm2rb1y/
✅ Copied: P50_cc45sm2y30.xyz → cc45sm2y/
✅ Copied: P50_cc45sm2y40.xyz → cc45sm2y/
✅ Copied: P50_cc45sm2y50.xyz → cc45sm2y/
✅ Copied: P50_cc45tay40.xyz → cc45tay/
✅ Copied: P50_cc45wdsm2rb1y40.xyz → cc45wdsm2rb1y/
✅ Copied: P50_cc45y18.xyz → cc45y/
✅ Copied: P50_cc45y30.xyz → cc45y/
✅ Copied: P50_cc45y40.xyz → cc45y/
✅ Copied: P50_cc45y50.xyz → cc45y/
✅ Copied: P50_cc85blsb2rb3y40.xyz → cc85blsb2rb3y/
✅ Copied: P50_cc85ndsb2rb3y40.xyz → cc85ndsb2rb3y/
✅ Copied: P50_cc85qd10sb2rb3y40.xyz → cc85qd10sb2rb3y/
✅ Copied: P50_cc85qd20sb2rb3y40.xyz → cc85qd20sb2rb3y/
✅ Copied: P50_cc85sb2y30.xyz → cc85sb2y/
✅ Copied: P50_cc85sb2y40.xyz → cc85sb2y/
